In [4]:
import os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize

import lightgbm as lgb  # on macOS run ./install.sh first
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

HERE = Path.cwd()
TRAIN_CSV = HERE / 'train.csv'
TEST_CSV  = HERE / 'test.csv'
SEEDS = [0, 1, 2]

def log(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)
def mape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(np.mean(np.abs((p - y) / y)) * 100.0)

## 1. Загрузка

In [5]:
COL_MAP = {
    'new_id':'new_id', 'Месяц':'month',
    'Дата открытия, категориальный':'open_cat',
    'Торговая площадь, категориальный':'area_cat',
    'Населенный пункт':'city', 'Регион':'region',
    'Численность населения':'pop', 'Количество домохозяйств':'households',
    'Трафик пеший, в час':'traf_walk', 'Трафик авто, в час':'traf_car',
    'Маркетплейсы, доставки, постаматы (100 м)':'mp',
    'Медицинские уч. и аптеки (300 м)':'med',
    'Школы (300 м)':'schools', 'Остановки (300 м)':'stops',
    'Продуктовые магазины (500 м)':'groc', 'Пятерочки (500 м)':'pyat',
    'Количество касс':'kassy', 'Флаг алкогольной лицензии':'alco',
    'РТО':'rto',
}
CAT_FEATS = ['open_cat','area_cat','city','region']
STORE_NUM_FEATS = ['pop','households','traf_walk','traf_car','mp','med',
                   'schools','stops','groc','pyat','kassy','alco']

train = pd.read_csv(TRAIN_CSV).rename(columns=COL_MAP)
train = train.sort_values(['new_id','month']).reset_index(drop=True)
test_ids = pd.read_csv(TEST_CSV)['new_id'].values
print('train:', train.shape, 'stores:', train.new_id.nunique())

train: (206150, 19) stores: 20615


## 2. Feature engineering

In [6]:
def build_features(train):
    wide = train.pivot(index='new_id', columns='month', values='rto')
    wide.columns = [f'm{int(c)}' for c in wide.columns]
    wide = wide.reset_index()
    store = (train.drop_duplicates('new_id').set_index('new_id')
                  [CAT_FEATS + STORE_NUM_FEATS])
    wide = wide.merge(store, left_on='new_id', right_index=True, how='left')

    oct_by_region  = train.loc[train.month==10].groupby('region')['rto'].mean().to_dict()
    oct_by_city    = train.loc[train.month==10].groupby('city')['rto'].mean().to_dict()
    mean_by_region = train.groupby('region')['rto'].mean().to_dict()

    rows = []
    for t in range(4, 12):
        T = t - 1
        avail = [f'm{i}' for i in range(1, T + 1)]
        hist = wide[avail].values.astype(np.float64)
        f = pd.DataFrame({'new_id': wide['new_id'].values, 'target_month': t, 'T': T})

        f['lag_1'] = hist[:, T - 1]
        f['lag_2'] = hist[:, T - 2] if T >= 2 else np.nan
        f['lag_3'] = hist[:, T - 3] if T >= 3 else np.nan
        f['lag_6'] = hist[:, T - 6] if T >= 6 else np.nan

        def stats(a, pref):
            f[f'{pref}_mean']   = a.mean(axis=1)
            f[f'{pref}_median'] = np.median(a, axis=1)
            f[f'{pref}_std']    = a.std(axis=1, ddof=0)
            f[f'{pref}_min']    = a.min(axis=1)
            f[f'{pref}_max']    = a.max(axis=1)
        stats(hist[:, max(0, T - 3):], 'roll3')
        if T >= 6: stats(hist[:, T - 6:], 'roll6')
        else:
            for s in ['mean','median','std','min','max']:
                f[f'roll6_{s}'] = np.nan
        stats(hist, 'rollall')

        midx = np.arange(1, T + 1, dtype=float)
        mm = midx.mean(); mv = ((midx - mm) ** 2).sum()
        if mv > 0:
            f['slope']     = ((hist - hist.mean(axis=1, keepdims=True)) * (midx - mm)).sum(axis=1) / mv
            lh = np.log1p(hist)
            f['slope_log'] = ((lh - lh.mean(axis=1, keepdims=True)) * (midx - mm)).sum(axis=1) / mv
        else:
            f['slope'] = 0.0; f['slope_log'] = 0.0
        if T >= 3:
            last3 = hist[:, T - 3:]
            m3 = np.arange(1, 4, dtype=float)
            m3m = m3.mean()
            m3v = ((m3 - m3m) ** 2).sum()
            f['slope_last3'] = ((last3 - last3.mean(axis=1, keepdims=True)) * (m3 - m3m)).sum(axis=1) / m3v
        else:
            f['slope_last3'] = 0.0

        if T >= 3:
            coefs = np.polyfit(midx, hist.T, deg=2)
            f['quad_a'] = coefs[0]; f['quad_b'] = coefs[1]
        else:
            f['quad_a'] = 0.0; f['quad_b'] = 0.0

        f['last_over_mean']   = f['lag_1'] / f['rollall_mean']
        f['last_over_median'] = f['lag_1'] / f['rollall_median']
        f['mom_1'] = hist[:, T - 1] / hist[:, T - 2] - 1.0 if T >= 2 else 0.0
        f['mom_2'] = hist[:, T - 2] / hist[:, T - 3] - 1.0 if T >= 3 else 0.0
        if T >= 2:
            ma = hist[:, 1:] / np.where(hist[:, :-1] == 0, np.nan, hist[:, :-1]) - 1.0
            f['mom_mean'] = np.nanmean(ma, axis=1); f['mom_std'] = np.nanstd(ma, axis=1)
        else:
            f['mom_mean'] = 0.0; f['mom_std'] = 0.0
        f['cv'] = f['rollall_std'] / f['rollall_mean']

        f['month_sin'] = np.sin(2 * np.pi * t / 12)
        f['month_cos'] = np.cos(2 * np.pi * t / 12)

        for c in CAT_FEATS + STORE_NUM_FEATS:
            f[c] = wide[c].values
        f['lag1_x_kassy'] = f['lag_1'] * wide['kassy'].values
        f['per_capita']   = f['rollall_mean'] / (wide['pop'].values + 1.0)

        f['region_oct_mean'] = wide['region'].map(oct_by_region).astype(float)
        f['city_oct_mean']   = wide['city'].map(oct_by_city).astype(float)
        f['region_mean']     = wide['region'].map(mean_by_region).astype(float)
        f['store_vs_region'] = f['lag_1'] / f['region_oct_mean']

        f['target'] = wide[f'm{t}'].values if t <= 10 else np.nan
        rows.append(f)
    return pd.concat(rows, ignore_index=True)

def encode_cats(df, cols, mapping=None):
    if mapping is None:
        mapping = {c: {v: i for i, v in enumerate(df[c].astype(str).unique())} for c in cols}
    out = df.copy()
    for c in cols:
        out[c] = out[c].astype(str).map(mapping[c]).fillna(-1).astype(np.int32)
    return out, mapping

feat_df = build_features(train)
feat_df, _ = encode_cats(feat_df, CAT_FEATS)

drop_cols = {'new_id','target','target_month'}
feat_cols = [c for c in feat_df.columns if c not in drop_cols]
cat_idx = [feat_cols.index(c) for c in CAT_FEATS]

df = feat_df.loc[feat_df['target_month'].between(4, 10)].reset_index(drop=True)
df_pred = feat_df.loc[feat_df['target_month'] == 11].reset_index(drop=True)
print('features:', len(feat_cols), 'train rows:', len(df), 'pred rows:', len(df_pred))

features: 56 train rows: 144305 pred rows: 20615


## 3. Обёртки для моделей

In [7]:
def lgb_mape(X_tr, y_tr_raw, X_val, y_val_raw, cat_idx, seed,
             w_tr=None, w_val=None, num_boost_round=4000, best_iter=None):
    if w_tr is None: w_tr = 1.0 / y_tr_raw
    params = dict(objective='regression', metric='mape',
                  num_leaves=127, min_data_in_leaf=50,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                  learning_rate=0.03, verbose=-1, seed=seed,
                  num_threads=os.cpu_count() or 4)
    dtr = lgb.Dataset(X_tr, label=y_tr_raw, weight=w_tr,
                      categorical_feature=cat_idx, free_raw_data=False)
    if best_iter is not None:
        return lgb.train(params, dtr, num_boost_round=best_iter)
    if w_val is None: w_val = 1.0 / y_val_raw
    dval = lgb.Dataset(X_val, label=y_val_raw, weight=w_val,
                       categorical_feature=cat_idx, reference=dtr, free_raw_data=False)
    return lgb.train(params, dtr, num_boost_round=num_boost_round,
                     valid_sets=[dval], valid_names=['val'],
                     callbacks=[lgb.early_stopping(200, verbose=False)])

def lgb_ratio(X_tr, r_tr, X_val, r_val, w_tr, w_val, cat_idx, seed,
              num_boost_round=4000, best_iter=None):
    params = dict(objective='regression', metric='rmse',
                  num_leaves=127, min_data_in_leaf=50,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                  learning_rate=0.03, verbose=-1, seed=seed,
                  num_threads=os.cpu_count() or 4)
    dtr = lgb.Dataset(X_tr, label=r_tr, weight=w_tr,
                      categorical_feature=cat_idx, free_raw_data=False)
    if best_iter is not None:
        return lgb.train(params, dtr, num_boost_round=best_iter)
    dval = lgb.Dataset(X_val, label=r_val, weight=w_val,
                       categorical_feature=cat_idx, reference=dtr, free_raw_data=False)
    return lgb.train(params, dtr, num_boost_round=num_boost_round,
                     valid_sets=[dval], valid_names=['val'],
                     callbacks=[lgb.early_stopping(200, verbose=False)])

def cat_log(X_tr, y_tr_log, X_val, y_val_log, cat_cols, seed,
            w_tr=None, iters=4000, best_iter=None):
    model = CatBoostRegressor(
        loss_function='RMSE', iterations=(best_iter or iters),
        learning_rate=0.03, depth=8, l2_leaf_reg=3.0, random_seed=seed,
        thread_count=os.cpu_count() or 4, verbose=False,
        early_stopping_rounds=(None if best_iter else 200),
    )
    if best_iter:
        model.fit(X_tr, y_tr_log, sample_weight=w_tr, cat_features=cat_cols, verbose=False)
    else:
        model.fit(X_tr, y_tr_log, sample_weight=w_tr, cat_features=cat_cols,
                  eval_set=(X_val, y_val_log), verbose=False)
    return model

## 4. OOF-стекинг по фолдам `t ∈ {7, 8, 9, 10}`

Для каждого `t` обучаемся на `target_month < t`, предсказываем на `target_month == t`.
На выходе — честные out-of-fold предсказания каждой модели для обучения стекера.

In [ ]:
FOLDS = [7, 8, 9, 10]
MODELS = ['lgb_mape_log', 'lgb_ratio_w', 'cat_log_w']
oof_logs = {name: np.zeros(len(df)) for name in MODELS}
best_iters = {name: [] for name in MODELS}

for t in FOLDS:
    log(f'--- fold t={t} ---')
    tr_mask = df['target_month'] < t
    va_mask = df['target_month'] == t
    d_tr = df.loc[tr_mask]; d_va = df.loc[va_mask]

    sw_tr = np.exp(-0.1 * (11 - d_tr['target_month'].values)).astype(np.float64)

    X_tr = d_tr[feat_cols].values.astype(np.float32)
    X_va = d_va[feat_cols].values.astype(np.float32)
    y_tr = d_tr['target'].values.astype(np.float64)
    y_va = d_va['target'].values.astype(np.float64)
    y_tr_log = np.log1p(y_tr); y_va_log = np.log1p(y_va)
    lag1_tr = d_tr['lag_1'].values; lag1_va = d_va['lag_1'].values
    r_tr = np.log(y_tr / lag1_tr); r_va = np.log(y_va / lag1_va)
    w_tr_y = (1.0 / y_tr) * sw_tr
    w_va_y = 1.0 / y_va
    va_idx = d_va.index.values

    preds = []
    for s in SEEDS:
        m = lgb_mape(X_tr, y_tr, X_va, y_va, cat_idx, s, w_tr=w_tr_y, w_val=w_va_y)
        best_iters['lgb_mape_log'].append(m.best_iteration)
        preds.append(np.log(np.clip(m.predict(X_va), 1.0, None)))
    oof_logs['lgb_mape_log'][va_idx] = np.median(np.stack(preds), axis=0)
    log(f"  M1 lgb_mape_log  MAPE={mape(y_va, np.exp(oof_logs['lgb_mape_log'][va_idx])):.4f}")


    preds = []
    for s in SEEDS:
        m = lgb_ratio(X_tr, r_tr, X_va, r_va, w_tr_y, w_va_y, cat_idx, s)
        best_iters['lgb_ratio_w'].append(m.best_iteration)
        preds.append(np.log(np.clip(lag1_va * np.exp(m.predict(X_va)), 1.0, None)))
    oof_logs['lgb_ratio_w'][va_idx] = np.median(np.stack(preds), axis=0)
    log(f"  M2 lgb_ratio_w   MAPE={mape(y_va, np.exp(oof_logs['lgb_ratio_w'][va_idx])):.4f}")

    # M3: CatBoost log-RMSE с row-weights
    preds = []
    for s in SEEDS:
        m = cat_log(d_tr[feat_cols], y_tr_log, d_va[feat_cols], y_va_log,
                    CAT_FEATS, s, w_tr=sw_tr)
        best_iters['cat_log_w'].append(m.tree_count_)
        preds.append(np.log(np.clip(np.expm1(m.predict(d_va[feat_cols])), 1.0, None)))
    oof_logs['cat_log_w'][va_idx] = np.median(np.stack(preds), axis=0)
    log(f"  M3 cat_log_w     MAPE={mape(y_va, np.exp(oof_logs['cat_log_w'][va_idx])):.4f}")

[12:18:59] --- fold t=7 ---
[12:19:00]   M1 lgb_mape_log  MAPE=34.4712
[12:19:01]   M2 lgb_ratio_w   MAPE=13.1236
[12:19:44]   M3 cat_log_w     MAPE=5.7005
[12:19:44] --- fold t=8 ---
[12:19:45]   M1 lgb_mape_log  MAPE=34.8699
[12:19:46]   M2 lgb_ratio_w   MAPE=5.3842
[12:20:09]   M3 cat_log_w     MAPE=7.0268
[12:20:09] --- fold t=9 ---
[12:20:11]   M1 lgb_mape_log  MAPE=35.9000
[12:20:12]   M2 lgb_ratio_w   MAPE=8.1933
[12:22:05]   M3 cat_log_w     MAPE=7.1007
[12:22:05] --- fold t=10 ---
[12:22:14]   M1 lgb_mape_log  MAPE=13.6459
[12:22:17]   M2 lgb_ratio_w   MAPE=9.6213
[12:24:24]   M3 cat_log_w     MAPE=6.8467


## 5. Обучение стекера (multi-start Nelder-Mead) и подбор clipping

In [9]:
y_oof = df['target'].values.astype(np.float64)
lag1_oof = df['lag_1'].values
oof_stack = np.stack([oof_logs[m] for m in MODELS], axis=0)

def fit_stacker(oof_log_preds, y_true, lag1, lo, hi):
    n = oof_log_preds.shape[0]
    def obj(w):
        w = np.abs(w); w = w / (w.sum() + 1e-12)
        pred = np.exp(w @ oof_log_preds)
        pred = np.clip(pred, lo * lag1, hi * lag1)
        return mape(y_true, pred)
    best_w, best_v = None, np.inf
    starts = [np.full(n, 1 / n)] + [np.eye(n)[i] + 0.05 for i in range(n)]
    for s in starts:
        res = minimize(obj, s, method='Nelder-Mead',
                       options=dict(xatol=1e-5, fatol=1e-5, maxiter=800))
        if res.fun < best_v:
            best_v, best_w = res.fun, res.x
    w = np.abs(best_w); w /= w.sum()
    return w, best_v

best = (None, None, None, np.inf)
for lo in [0.4, 0.6, 0.7, 0.8]:
    for hi in [1.3, 1.4, 1.5, 1.8, 2.5]:
        w, v = fit_stacker(oof_stack, y_oof, lag1_oof, lo, hi)
        if v < best[3]:
            best = (w, lo, hi, v)
w, lo, hi, best_mape = best
print('stacker weights:', dict(zip(MODELS, w.round(3))))
print(f'clip=[{lo}, {hi}], OOF MAPE = {best_mape:.4f}')
print(f'expected score ≈ {100 - min(best_mape, 100):.2f}')

# отдельно оценим на t=10 фолде
m10 = df['target_month'].values == 10
pred10 = np.exp(w @ oof_stack[:, m10])
pred10 = np.clip(pred10, lo * lag1_oof[m10], hi * lag1_oof[m10])
print(f't=10 fold MAPE = {mape(y_oof[m10], pred10):.4f}')

stacker weights: {'lgb_mape_log': np.float64(0.002), 'lgb_ratio_w': np.float64(0.077), 'cat_log_w': np.float64(0.921)}
clip=[0.8, 2.5], OOF MAPE = 13.2862
expected score ≈ 86.71
t=10 fold MAPE = 7.0510


## 6. Финальное обучение

In [10]:
X_full = df[feat_cols].values.astype(np.float32)
y_full = df['target'].values.astype(np.float64)
y_full_log = np.log1p(y_full)
lag1_full = df['lag_1'].values
r_full = np.log(y_full / lag1_full)
sw_full = np.exp(-0.1 * (11 - df['target_month'].values)).astype(np.float64)
w_full_y = (1.0 / y_full) * sw_full

X_pred = df_pred[feat_cols].values.astype(np.float32)
lag1_pred = df_pred['lag_1'].values

# используем медиану best_iteration из OOF — нет eval_set leak
best_iter_m1 = int(np.median(best_iters['lgb_mape_log']))
best_iter_m2 = int(np.median(best_iters['lgb_ratio_w']))
best_iter_m3 = int(np.median(best_iters['cat_log_w']))
print(f'best_iters: M1={best_iter_m1}, M2={best_iter_m2}, M3={best_iter_m3}')

# M1
preds = []
for s in SEEDS:
    log(f'  M1 seed={s}')
    m = lgb_mape(X_full, y_full, None, None, cat_idx, s, w_tr=w_full_y, best_iter=best_iter_m1)
    preds.append(np.log(np.clip(m.predict(X_pred), 1.0, None)))
log_p1 = np.median(np.stack(preds), axis=0)

# M2
preds = []
for s in SEEDS:
    log(f'  M2 seed={s}')
    m = lgb_ratio(X_full, r_full, None, None, w_full_y, None, cat_idx, s, best_iter=best_iter_m2)
    preds.append(np.log(np.clip(lag1_pred * np.exp(m.predict(X_pred)), 1.0, None)))
log_p2 = np.median(np.stack(preds), axis=0)

# M3
preds = []
for s in SEEDS:
    log(f'  M3 seed={s}')
    m = cat_log(df[feat_cols], y_full_log, None, None, CAT_FEATS, s, w_tr=sw_full, best_iter=best_iter_m3)
    preds.append(np.log(np.clip(np.expm1(m.predict(df_pred[feat_cols])), 1.0, None)))
log_p3 = np.median(np.stack(preds), axis=0)

log_stack_pred = np.stack([log_p1, log_p2, log_p3], axis=0)
pred_nov = np.exp(w @ log_stack_pred)
pred_nov = np.clip(pred_nov, lo * lag1_pred, hi * lag1_pred)
print('pred_nov stats:', pred_nov.mean(), pred_nov.min(), pred_nov.max())

best_iters: M1=1, M2=1, M3=966
[12:24:31]   M1 seed=0
[12:24:32]   M1 seed=1
[12:24:32]   M1 seed=2
[12:24:32]   M2 seed=0
[12:24:32]   M2 seed=1
[12:24:33]   M2 seed=2
[12:24:33]   M3 seed=0
[12:25:07]   M3 seed=1
[12:25:44]   M3 seed=2
pred_nov stats: 34524223.79575385 5500884.217288531 159165346.7973744


## 7. Submission

In [11]:
by_id = dict(zip(df_pred['new_id'].values, pred_nov))
out = np.array([by_id[i] for i in test_ids], dtype=np.float64)
sub = pd.DataFrame({'new_id': test_ids, 'rto': out})

assert len(sub) == 20615
assert list(sub.columns) == ['new_id','rto']
assert sub.rto.isna().sum() == 0 and sub.rto.min() > 0 and sub.rto.max() < 5e9

oct_mean = train.loc[train.month == 10, 'rto'].mean()
print(f"sub: mean={sub.rto.mean():.0f}, median={sub.rto.median():.0f}, "
      f"min={sub.rto.min():.0f}, max={sub.rto.max():.0f}")
print(f"oct_mean={oct_mean:.0f}, rel={abs(sub.rto.mean()-oct_mean)/oct_mean:.3f}")

sub.to_csv(TEST_CSV, index=False)
print('wrote', TEST_CSV)
print(f'expected score ≈ {100 - min(best_mape, 100):.2f}')

sub: mean=34524224, median=29464501, min=5500884, max=159165347
oct_mean=33503980, rel=0.030
wrote /Users/alex/ML Projects/HSE/test.csv
expected score ≈ 86.71
